# Fine-tuning

All transformer models in this project are trained by a single routine (`src/train.py`)
under one fixed protocol, so their scores are directly comparable:

- **Constant effective batch (32).** The per-device batch and gradient-accumulation
  adapt to the number of GPUs, but the number of gradient updates is identical on a
  single T4 (Colab) or dual T4 (Kaggle). This removes the multi-GPU confound that
  previously invalidated the sequence-length comparison.
- **5 epochs, LR 2e-5, 6% warmup, weight decay 0.01**, regression head, best
  checkpoint chosen by dev Spearman.
- A **timing probe** estimates the full-run cost from the first 200 steps before any
  long run is committed.

The ladder of models: DistilBERT (66M) → RoBERTa-base (125M) → RoBERTa-large (355M).
This lets us measure how humor-score prediction scales with model capacity against the
rJokes paper's ~340M-param baselines.

Reported test numbers come from `05_evaluation.ipynb`, not this notebook, so there is
one source of truth for every figure in the report.


In [ ]:
# Setup
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/humor-intelligence.git"

!pip install -q -U "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30" scipy scikit-learn

import os, sys
if not os.path.exists("humor-intelligence"):
    !git clone -q $GITHUB_REPO_URL
REPO_DIR = os.path.abspath("humor-intelligence")
!cd "$REPO_DIR" && python src/data.py
sys.path.append(os.path.join(REPO_DIR, "src"))

# Hugging Face auth (Kaggle secret named HF_TOKEN, else interactive)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# Run configurations for the model ladder
from train import RunConfig, train

RUNS = {
    "distilbert-128":    RunConfig("distilbert-128",    "distilbert-base-uncased",
                                   hf_repo="iamahmadyasin/humor-distilbert"),
    "roberta-base-128":  RunConfig("roberta-base-128",  "roberta-base",
                                   hf_repo="iamahmadyasin/humor-roberta-base"),
    "roberta-large-128": RunConfig("roberta-large-128", "roberta-large",
                                   per_device_override=8,
                                   hf_repo="iamahmadyasin/humor-roberta-large"),
}

cfg = RUNS["roberta-base-128"]

In [ ]:
# Timing probe: estimate the full-run cost before committing GPU hours
train(cfg, probe_steps=200)

In [ ]:
# Full run. On Kaggle, launch with Save & Run All (Commit) so it runs server-side and survives disconnects. Resumes automatically if restarted.
train(cfg, push=True)